In [1]:
import pandas as pd
import re
import unicodedata
from itables import show

# Ingesta de datos

In [2]:
# Ponemos la ruta donde se encuentra cada uno de los archivos
# el segundo arhcivo de cada uno de los DF refiere a la actualización de fechas

df_costeos_serv_raw_1 = pd.read_csv('C:/Users/Laura/OneDrive - MOLT/Documentos/Proveedores y clientes/response.csv')
df_costeos_serv_raw_2 = pd.read_csv('C:/Users/Laura/OneDrive - MOLT/Documentos/Proveedores y clientes/response2.csv')
df_costeos_serv_raw = pd.concat([df_costeos_serv_raw_1, df_costeos_serv_raw_2], ignore_index=True)

# mapa de servicios id
df_servicios_id = pd.read_csv('C:/Users/Laura/OneDrive - MOLT/Documentos/Proveedores y clientes/Todos Procesos Producción.csv')

# Costeos x PTO
df_costeos_pto1 = pd.read_csv('C:/Users/Laura/OneDrive - MOLT/Documentos/Proveedores y clientes/Todos Costeo Producto.csv')
df_costeos_pto2 = pd.read_csv('C:/Users/Laura/OneDrive - MOLT/Documentos/Proveedores y clientes/Todos Costeo Producto (1).csv')
df_costeos_pto = pd.concat([df_costeos_pto1, df_costeos_pto2], ignore_index=True)


# OP detalle
df_op_det1 = pd.read_csv('C:/Users/Laura/OneDrive - MOLT/Documentos/Proveedores y clientes/File.csv')
df_op_det2 = pd.read_csv('C:/Users/Laura/OneDrive - MOLT/Documentos/Proveedores y clientes/File (1).csv')
df_op_det = pd.concat([df_op_det1, df_op_det2], ignore_index=True)

# Ordenes de Servicio
df_os1 = pd.read_csv('C:/Users/Laura/OneDrive - MOLT/Documentos/Proveedores y clientes/Todas OS.csv')
df_os2 = pd.read_csv('C:/Users/Laura/OneDrive - MOLT/Documentos/Proveedores y clientes/Todas OS (1).csv')
df_os = pd.concat([df_os1, df_os2], ignore_index=True)


#Comercial
df_comercial1 = pd.read_csv('C:/Users/Laura/OneDrive - MOLT/Documentos/Proveedores y clientes/Todos Costeos.csv')
df_comercial2 = pd.read_csv('C:/Users/Laura/OneDrive - MOLT/Documentos/Proveedores y clientes/Todos Costeos (1).csv')
df_comercial = pd.concat([df_comercial1, df_comercial2], ignore_index=True)

# OP (Solo para tomar el valor total de la OP)
df_op1 = pd.read_csv('C:/Users/Laura/OneDrive - MOLT/Documentos/Proveedores y clientes/Todas OP.csv')
df_op2 = pd.read_csv('C:/Users/Laura/OneDrive - MOLT/Documentos/Proveedores y clientes/Todas OP_1.csv')
df_op = pd.concat([df_op1, df_op2], ignore_index=True)

### Actualización de fechas

In [3]:
df_costeos_serv_raw

,Unnamed: 0,0,1,2,3
0,0.0,81866890.0,15339263,11000.0,2026-03-17T14:56:23Z
1,1.0,81866890.0,18429545,9000.0,2026-03-17T14:54:39Z
2,2.0,81865874.0,15140099,300.0,2026-03-17T14:32:25Z
3,3.0,81865874.0,16093810,800.0,2026-03-17T14:32:25Z
4,4.0,81865874.0,16538227,113000.0,2026-03-17T14:32:25Z
...,...,...,...,...,...
339271,NaN,79946417.0,16544003,2000.0,2026-02-11T13:34:18Z
339272,NaN,79946407.0,16093810,800.0,2026-02-11T13:34:18Z
339273,NaN,79946407.0,18429545,8500.0,2026-02-11T13:34:18Z
339274,NaN,79946407.0,16544003,2000.0,2026-02-11T13:34:18Z


In [3]:
df_costeos_serv_raw.columns
df_costeos_serv_raw = df_costeos_serv_raw.drop('Unnamed: 0', axis=1)

In [4]:
df_costeos_serv_raw.columns = ['id_costeoxpto', 'id_serv', 'costo_antes_iva', 'fecha_creacion']

In [5]:
df_costeos_serv_raw.shape

(329277, 4)

In [6]:
df_costeos_serv_raw.describe()

,id_costeoxpto,id_serv,costo_antes_iva
count,3.291510e+05,3.292770e+05,3.292720e+05
mean,4.187977e+07,1.692871e+07,1.003151e+04
std,1.592123e+07,1.516889e+06,2.265866e+05
min,1.511720e+07,1.415802e+07,0.000000e+00
25%,2.615014e+07,1.653823e+07,1.400000e+03
50%,4.132040e+07,1.654400e+07,3.000000e+03
75%,5.405768e+07,1.842954e+07,9.000000e+03
max,8.186689e+07,4.537002e+07,4.096208e+07


In [7]:
df_costeos_serv_raw = df_costeos_serv_raw.dropna(axis=0, subset=['id_costeoxpto'])
df_costeos_serv_raw = df_costeos_serv_raw.dropna(axis=0, subset=['costo_antes_iva'])

In [8]:
df_costeos_serv_raw = df_costeos_serv_raw.drop_duplicates()

In [9]:
df_costeos_serv_raw.shape

(119125, 4)

# Data Cleaning

In [10]:
# Homologación servicios costeos
df_costeos_serv = df_costeos_serv_raw.merge(
    df_servicios_id[["Id.", "Proceso Producción"]].rename(columns={"Id.": "id_serv"}),
    on="id_serv",
    how="left",
).drop(columns=["id_serv"])

# union con costeos x pto
df_costeos_pto = df_costeos_pto[
    ["Costeo_Producto", "Producto", "Cliente", "Cantidad Solicitada", "Id.", "Fecha"]
].rename(columns={"Producto":"Productos"})

df_costeos_pto = df_costeos_pto.merge(
    df_costeos_serv.rename(columns={"id_costeoxpto": "Id."}),
    on="Id.",
    how="left",
)

df_costeos_pto

,Costeo_Producto,Productos,Cliente,Cantidad Solicitada,Id.,Fecha,costo_antes_iva,fecha_creacion,Proceso Producción
0,COS-0012287-2,Chaqueta Térmica,Coorserpark,34,77898260,30/12/2025,1800.0,2025-12-30T12:41:33Z,Bordado
1,COS-0012287-2,Chaqueta Térmica,Coorserpark,34,77898260,30/12/2025,18000.0,2025-12-30T12:41:33Z,Confección
2,COS-0012288-1,CHALECO REFLECTIVO,TALMA COLOMBIA,2,77917786,30/12/2025,1500.0,2025-12-31T00:42:03Z,Estampado
3,COS-0012288-1,CHALECO REFLECTIVO,TALMA COLOMBIA,2,77917786,30/12/2025,12000.0,2025-12-31T00:39:59Z,Confección
4,COS-0012287-2,Gorra Curva,Coorserpark,4,77898321,30/12/2025,1800.0,2025-12-30T12:43:54Z,Bordado
...,...,...,...,...,...,...,...,...,...
87553,COS-0012289-2,POLO MANGA LARGA,Molt Panamá (PA),35,77979670,02/01/2026,1750.0,2026-01-02T20:27:18Z,Bordado
87554,COS-0012289-2,POLO MANGA LARGA,Molt Panamá (PA),35,77979753,02/01/2026,1800.0,2026-01-02T20:29:53Z,Estampado
87555,COS-0012289-2,POLO MANGA LARGA,Molt Panamá (PA),35,77979753,02/01/2026,11000.0,2026-01-02T20:29:53Z,Transporte
87556,COS-0012289-2,POLO MANGA LARGA,Molt Panamá (PA),35,77979753,02/01/2026,7500.0,2026-01-02T20:29:53Z,Confección


In [11]:
df_costeos_pto.duplicated().sum()

np.int64(0)

In [12]:
df_op_det.duplicated().sum()

np.int64(0)

In [13]:
# unir a OP detalle
df_op_det.drop(
    columns=[
        "ClienteNombre",
        "Categoría Proc",
        "Detalle",
        "Colores",
        "Total Pieza",
        "Cantidad Remitida",
        "Precio Unitario Sin IVA",
        "Estado OP",
        "Fecha Promesa",
        "Ficha Técnica Producto",
        "Muestra",
        "Id.",
        "OS",
        "Creado El"
    ],
    inplace=True,
)
df_op_det = df_op_det.rename(columns={'Id Costeo Producto' : 'Id.'})

In [14]:
df_op_det = df_op_det.drop_duplicates()
df_op_det.duplicated().sum()

np.int64(0)

In [15]:
df_op_det = df_op_det.merge(
    df_costeos_pto,
    on=["Costeo_Producto", "Productos", "Id."],
    how="left",
)
df_op_det

,Num-OP,Secuencia,OP Det,Productos,Producir,Total Precio Sin IVA,Id.,Costeo_Producto,Cliente,Cantidad Solicitada,Fecha,costo_antes_iva,fecha_creacion,Proceso Producción
0,6672.0,1.0,6672-1-Buzo / Cerrado,Buzo / Cerrado,26,1559168.0,81884540.0,COS-0012801-2,GRAN AMERICA,1.0,17/03/2026,NaN,NaN,NaN
1,6672.0,2.0,6672-2-Buzo / Cerrado,Buzo / Cerrado,26,1559168.0,81884548.0,COS-0012801-3,GRAN AMERICA,1.0,17/03/2026,NaN,NaN,NaN
2,6672.0,3.0,6672-3-Buzo / Cerrado,Buzo / Cerrado,36,2158848.0,81884555.0,COS-0012801-4,GRAN AMERICA,1.0,17/03/2026,NaN,NaN,NaN
3,6672.0,4.0,6672-4-Buzo / Cerrado,Buzo / Cerrado,36,2158848.0,81884562.0,COS-0012801-5,GRAN AMERICA,1.0,17/03/2026,NaN,NaN,NaN
4,6671.0,1.0,6671-1-Abrigo Sastre,Abrigo Sastre,1,522104.0,81147999.0,COS-0012713-1,AVIANCA,2330.0,04/03/2026,300.0,2026-03-09T16:44:21Z,Empaque
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18686,4323.0,2.0,4323-2,T-SHIRT MANGA CORTA,40,1434440.0,38221399.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
18687,4323.0,3.0,4323-3,T-SHIRT MANGA CORTA,40,1434440.0,38221403.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
18688,4322.0,1.0,4322-1,HODDIE CERRADO,27,1735560.0,38211312.0,COS-0006031-2,HEEL,27.0,10/01/2023,1000.0,2023-01-10T18:10:53Z,Estampado
18689,4322.0,1.0,4322-1,HODDIE CERRADO,27,1735560.0,38211312.0,COS-0006031-2,HEEL,27.0,10/01/2023,1400.0,2023-01-10T18:10:53Z,Bordado


In [16]:
df_op_det = df_op_det.dropna(axis=0, subset=['costo_antes_iva'])

In [17]:
df_op_det

,Num-OP,Secuencia,OP Det,Productos,Producir,Total Precio Sin IVA,Id.,Costeo_Producto,Cliente,Cantidad Solicitada,Fecha,costo_antes_iva,fecha_creacion,Proceso Producción
4,6671.0,1.0,6671-1-Abrigo Sastre,Abrigo Sastre,1,522104.0,81147999.0,COS-0012713-1,AVIANCA,2330.0,04/03/2026,300.0,2026-03-09T16:44:21Z,Empaque
5,6671.0,1.0,6671-1-Abrigo Sastre,Abrigo Sastre,1,522104.0,81147999.0,COS-0012713-1,AVIANCA,2330.0,04/03/2026,800.0,2026-03-09T13:57:03Z,Transporte
6,6671.0,1.0,6671-1-Abrigo Sastre,Abrigo Sastre,1,522104.0,81147999.0,COS-0012713-1,AVIANCA,2330.0,04/03/2026,98000.0,2026-03-09T13:56:47Z,Corte y Confeccion
7,6671.0,2.0,6671-2-Abrigo Sastre,Abrigo Sastre,1,537035.0,81148032.0,COS-0012713-2,AVIANCA,1101.0,04/03/2026,300.0,2026-03-09T16:45:53Z,Empaque
8,6671.0,2.0,6671-2-Abrigo Sastre,Abrigo Sastre,1,537035.0,81148032.0,COS-0012713-2,AVIANCA,1101.0,04/03/2026,800.0,2026-03-09T14:00:14Z,Transporte
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18683,4324.0,1.0,4324-1,Camisa trabajo,145,20686715.0,38247007.0,COS-0006041-12,ZX VENTURE COLOMBIA SAS,1.0,12/01/2023,11200.0,2023-01-12T18:01:55Z,Corte y Confeccion
18684,4324.0,2.0,4324-2,BOC074 BOTA CAUCHO PUNTERA,36,2307312.0,38247026.0,COS-0006041-12,ZX VENTURE COLOMBIA SAS,36.0,12/01/2023,49000.0,2023-01-12T18:04:58Z,Paquete Completo
18688,4322.0,1.0,4322-1,HODDIE CERRADO,27,1735560.0,38211312.0,COS-0006031-2,HEEL,27.0,10/01/2023,1000.0,2023-01-10T18:10:53Z,Estampado
18689,4322.0,1.0,4322-1,HODDIE CERRADO,27,1735560.0,38211312.0,COS-0006031-2,HEEL,27.0,10/01/2023,1400.0,2023-01-10T18:10:53Z,Bordado


In [18]:
# Limpieza de comercial

df_comercial.columns

# Eliminación de columnas no relevantes
df_comercial.drop(
    columns=[
        "Nombre",
        "Estado",
        "Margen Ponderado",
        "Creado en",
        "Acciones",
    ],
    inplace=True,
)
df_comercial.head()


,Costeo,Cliente,Nombre del Comercial,Id.
0,COS-0012876,AEROSAN S.A.S.,Pilar Gómez,82524713
1,COS-0012875,NUTRESA,Luis Guzman,82520897
2,COS-0012874,NUTRESA,Luis Guzman,82496746
3,COS-0012873,NUTRESA,Luis Guzman,82495472
4,COS-0012872,NUTRESA,Luis Guzman,82492851


In [19]:
df_op_det['Num-OP'] = pd.to_numeric(df_op_det['Num-OP'], errors='coerce').fillna(0).astype(int)
df_op_det['Secuencia'] = pd.to_numeric(df_op_det['Secuencia'], errors='coerce').fillna(0).astype(int)

df_op_det['OP-D'] = df_op_det['Num-OP'].astype(str) + '-' + df_op_det['Secuencia'].astype(str)

df_op_det['secuencia'] = df_op_det['Secuencia']

C:\Users\Laura\AppData\Local\Temp\ipykernel_30944\2285874593.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_op_det['Num-OP'] = pd.to_numeric(df_op_det['Num-OP'], errors='coerce').fillna(0).astype(int)
C:\Users\Laura\AppData\Local\Temp\ipykernel_30944\2285874593.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_op_det['Secuencia'] = pd.to_numeric(df_op_det['Secuencia'], errors='coerce').fillna(0).astype(int)
C:\Users\Laura\AppData\Local\Temp\ipykernel_30944\2285874593.py:4: SettingWithCopyWarnin

In [20]:
df_op_det

,Num-OP,Secuencia,OP Det,Productos,Producir,Total Precio Sin IVA,Id.,Costeo_Producto,Cliente,Cantidad Solicitada,Fecha,costo_antes_iva,fecha_creacion,Proceso Producción,OP-D,secuencia
4,6671,1,6671-1-Abrigo Sastre,Abrigo Sastre,1,522104.0,81147999.0,COS-0012713-1,AVIANCA,2330.0,04/03/2026,300.0,2026-03-09T16:44:21Z,Empaque,6671-1,1
5,6671,1,6671-1-Abrigo Sastre,Abrigo Sastre,1,522104.0,81147999.0,COS-0012713-1,AVIANCA,2330.0,04/03/2026,800.0,2026-03-09T13:57:03Z,Transporte,6671-1,1
6,6671,1,6671-1-Abrigo Sastre,Abrigo Sastre,1,522104.0,81147999.0,COS-0012713-1,AVIANCA,2330.0,04/03/2026,98000.0,2026-03-09T13:56:47Z,Corte y Confeccion,6671-1,1
7,6671,2,6671-2-Abrigo Sastre,Abrigo Sastre,1,537035.0,81148032.0,COS-0012713-2,AVIANCA,1101.0,04/03/2026,300.0,2026-03-09T16:45:53Z,Empaque,6671-2,2
8,6671,2,6671-2-Abrigo Sastre,Abrigo Sastre,1,537035.0,81148032.0,COS-0012713-2,AVIANCA,1101.0,04/03/2026,800.0,2026-03-09T14:00:14Z,Transporte,6671-2,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18683,4324,1,4324-1,Camisa trabajo,145,20686715.0,38247007.0,COS-0006041-12,ZX VENTURE COLOMBIA SAS,1.0,12/01/2023,11200.0,2023-01-12T18:01:55Z,Corte y Confeccion,4324-1,1
18684,4324,2,4324-2,BOC074 BOTA CAUCHO PUNTERA,36,2307312.0,38247026.0,COS-0006041-12,ZX VENTURE COLOMBIA SAS,36.0,12/01/2023,49000.0,2023-01-12T18:04:58Z,Paquete Completo,4324-2,2
18688,4322,1,4322-1,HODDIE CERRADO,27,1735560.0,38211312.0,COS-0006031-2,HEEL,27.0,10/01/2023,1000.0,2023-01-10T18:10:53Z,Estampado,4322-1,1
18689,4322,1,4322-1,HODDIE CERRADO,27,1735560.0,38211312.0,COS-0006031-2,HEEL,27.0,10/01/2023,1400.0,2023-01-10T18:10:53Z,Bordado,4322-1,1


In [21]:
df_costeos_serv_raw.columns

Index(['id_costeoxpto', 'id_serv', 'costo_antes_iva', 'fecha_creacion'], dtype='object')

In [22]:
df_os.duplicated().sum()

np.int64(0)

In [ ]:
# añadir prefijo OS a columnas de df_os
for col in df_os.columns:
    if col not in ['Num OS', 'Orden de Satélite', 'Satélite Procesos']:
        df_os.rename(columns={col: 'OS-' + col}, inplace=True)

df_os['OP-D'] = df_os['Orden de Satélite'].str.split('-').str[2] + '-' + df_os['Orden de Satélite'].str.split('-').str[3]

#Ponemos el "NUM OS" dentro del groupby para que no se pierna la información
df_os_grouped = df_os.groupby(['OP-D', 'Satélite Procesos', 'OS-Proveedor', 'Num OS']).agg({
    'OS-Valor Unitario': 'mean',
    'OS-Cantidad': 'sum'
}).reset_index()

In [24]:
df_op_det.shape

(16688, 16)

In [25]:
# Costeo servicio op vs os
df_op_det_vs_os = df_op_det.merge(
    df_os_grouped.rename(columns={"Satélite Procesos": "Proceso Producción"}),
    on=["OP-D", "Proceso Producción"],
    how="left",
)


In [26]:
# Filtramos para quedarnos solo con las filas que no esten vacias en OS-Valor Unitario 
df_op_det_vs_os = df_op_det_vs_os[df_op_det_vs_os['OS-Valor Unitario'].notna()]

# Separamos el último número del costeo para poder unir con el df de comercial
df_op_det_vs_os['Costeo'] = (df_op_det_vs_os['Costeo_Producto'].str.rsplit('-', n=1).str[0])


In [27]:
df_comercial.duplicated().sum()

np.int64(0)

In [28]:
df_final = df_op_det_vs_os.merge(
    df_comercial,
    on=["Costeo", "Cliente"],
    how="left"
)
df_final.duplicated().sum()

np.int64(0)

In [29]:
df_final = df_final[[
    'Num-OP', 'Secuencia', 'OP-D', 'OP Det', 'Productos', 'Cliente', 
    'Fecha', 'Id._x', 'Costeo_Producto', 'Costeo', 'Proceso Producción', 
    'Producir', 'OS-Cantidad', 'Num OS',  'Nombre del Comercial','OS-Proveedor',
    'costo_antes_iva', 'OS-Valor Unitario', 
]].rename(columns={'Id._x' : 'Id.'})
df_final.drop_duplicates(inplace=True)


In [30]:
df_op = df_op[[
    'Num-OP','Estado','Total Precio OP' ]]
df_op.drop_duplicates(inplace=True)

In [32]:
df_fin = df_final.merge(
    df_op,
    on="Num-OP",
    how="left"
)
df_fin.duplicated().sum()

np.int64(0)

In [33]:
# Definimos la lista de categorías oficiales para la clasificación de productos
categorias_oficiales = [
    'Accesorios', 'Bata', 'Bermuda', 'Blusa', 'Bolígrafo', 'Botas', 'Botella', 'Botón', 'Broche', 'Buzo', 'Camibuzo', 
    'Camiseta', 'Camisa', 'Cachuchera', 'Casco', 'Chaleco', 'Chaqueta', 'Cinta', 'Cofia', 'Conjunto', 'Cordón', 
    'Cremallera', 'Cuello', 'Delantal', 'Diseño', 'Embone', 'Falda', 'Gafas seguridad', 'Gorra', 'Gorro', 'Guantes', 
    'Guata', 'Hilo', 'Hoodie', 'Impermeable', 'Jean', 'Libreta', 'Lonchera', 'Mangas', 'Marquilla', 'Máscara', 'Medias', 
    'Morral', 'Ojalete', 'Overol', 'Pantalón', 'Pantaloneta', 'Pañoleta', 'Paraguas', 'Pavas', 'Polainas', 'Resorte', 
    'Ropa interior', 'Saco', 'Sastre', 'Servicios', 'Sesgo', 'Sudadera', 'Suéter', 'Tankas', 'Tapabocas', 
    'Tapaoídos', 'Tecnología', 'Termo', 'Uniformes', 'Velcro', 'Zapatos', 'Escafandra'
]

# Creamos una función para normalizar el texto, eliminando acentos y convirtiendo a minúsculas
def normalizar(texto):
    texto = str(texto).lower()
    texto = unicodedata.normalize('NFKD', texto).encode('ascii', 'ignore').decode('utf-8')
    return texto

# Mapeos especiales: palabras clave -> categoría oficial
MAPEOS_ESPECIALES = {
    'polo':       'Camiseta',
    'camibuso':   'Camibuzo',   # cubre "camibuso" (mal escrito) -> "Camibuzo"
    'harrington': 'Chaqueta',
    'termica':    'Chaqueta',   # cubre "térmica" también (por la normalización)
    'hoddie':     'Hoodie',     # cubre el typo "hoddie" -> "Hoodie"
    'jogger':     'Pantalón',
}

def clasificar(Producto):
    producto_norm = normalizar(Producto)
    
    # 1. Primero revisamos los mapeos especiales
    for keyword, categoria in MAPEOS_ESPECIALES.items():
        pattern = r'\b' + re.escape(keyword) + r'\b'
        if re.search(pattern, producto_norm):
            return categoria
    
    # 2. Luego la búsqueda genérica en categorías oficiales
    for cat in categorias_oficiales:
        cat_norm = normalizar(cat)
        pattern = r'\b' + re.escape(cat_norm) + r'\b'
        if re.search(pattern, producto_norm):
            return cat
    
    return 'Otros'

df_final['Categoría'] = df_final['Productos'].apply(clasificar)

df_fin['Categoría'] = df_fin['Productos'].apply(clasificar)   

In [34]:
df_fin.columns

Index(['Num-OP', 'Secuencia', 'OP-D', 'OP Det', 'Productos', 'Cliente',
       'Fecha', 'Id.', 'Costeo_Producto', 'Costeo', 'Proceso Producción',
       'Producir', 'OS-Cantidad', 'Num OS', 'Nombre del Comercial',
       'OS-Proveedor', 'costo_antes_iva', 'OS-Valor Unitario', 'Estado',
       'Total Precio OP', 'Categoría'],
      dtype='object')

## Union con excel de producción

In [25]:
excell = pd.ExcelFile("SEGUIMIENTO PRODUCCION.xlsx")
df_produccion_confeccion = pd.read_excel(excell, sheet_name='Satelites', header=2)
df_produccion_bye = pd.read_excel(excell, sheet_name='Bordados-Estampados', header=2)
df_produccion_bye.columns

C:\Users\Laura\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)
C:\Users\Laura\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


Index(['ID', 'ESTADO', '# OS SÁTELITE CONFECCIÓN', '#OP DETALLE', 'CLIENTE',
       'DESCRIPCION PRENDA', 'CANTIDAD O.S.',
       'FECHA ENVIO A TALLER DE SERVICIO', 'FECHA COMPROMISO SERVICIO',
       'FECHA ENTREGA BORDADO Y/O ESTAMPADO', '# DIAS EN TALLER',
       'PRECIO MARCACIÓN', 'TALLER SERVICIO', 'CANTIDAD ENTREGA SATELITE',
       'CANTIDAD PENDIENTE SATELITE', 'SEXO', 'T:U', '28', '30', '32', '34',
       '36', '38', '40', '42', '44', '46', '48', '50', '52',
       'TALLER CONFECCION', 'Unnamed: 31', 'Unnamed: 32'],
      dtype='object')

In [26]:
df_produccion_confeccion = df_produccion_confeccion[[
    '#OP DETALLE',
    '# OS SÁTELITE CONFECCIÓN',
    'CANTIDAD ENTREGA SATELITE',
    'TALLER CONFECCION'
]]

df_produccion_confeccion = df_produccion_confeccion.rename(columns={
    '#OP DETALLE' : 'OP-D', 
    '# OS SÁTELITE CONFECCIÓN' : 'Num OS',
    'TALLER CONFECCION': 'TALLER SERVICIO'})

df_produccion_confeccion['Num OS'] = df_produccion_confeccion['Num OS'].astype('Int64')

df_produccion_confeccion

,OP-D,Num OS,CANTIDAD ENTREGA SATELITE,TALLER SERVICIO
0,4968-10,12447,3.0,FLIPPER SPORT
1,5140-1,12451,55.0,DIOMEDEZ MARTINEZ
2,5140-2,12452,11.0,DIOMEDEZ MARTINEZ
3,5140-3,12454,8.0,MAUDA ALFONSO
4,5140-4,12455,33.0,MAUDA ALFONSO
...,...,...,...,...
4892,6639-22,21675,NaN,URIEL RUIZ
4893,6639-23,21683,NaN,URIEL RUIZ
4894,6650-5,21320,NaN,PATRICIA HUERTAS
4895,6625-7,21694,NaN,MARISOL BAUTISTA


In [27]:
df_produccion_bye = df_produccion_bye[[
    '#OP DETALLE',
    '# OS SÁTELITE CONFECCIÓN',
    'CANTIDAD ENTREGA SATELITE',
    'TALLER SERVICIO'
]]

df_produccion_bye = df_produccion_bye.rename(columns={'#OP DETALLE' : 'OP-D', '# OS SÁTELITE CONFECCIÓN' : 'Num OS' })

df_produccion_bye['Num OS'] = df_produccion_bye['Num OS'].astype('Int64')

df_produccion_bye = df_produccion_bye.dropna(how='all')

df_produccion = pd.concat([
    df_produccion_bye, df_produccion_confeccion
    ], ignore_index=True)

df_produccion

,OP-D,Num OS,CANTIDAD ENTREGA SATELITE,TALLER SERVICIO
0,5182-1,12642,200.0,DESIGNS AND PRINTS
1,5177-3,12647,20.0,SEIIO
2,5182-3,12650,500.0,DESIGNS AND PRINTS
3,5182-3,12651,500.0,DESIGNS AND PRINTS
4,5197-1,12652,35.0,SEIIO
...,...,...,...,...
9065,6639-22,21675,NaN,URIEL RUIZ
9066,6639-23,21683,NaN,URIEL RUIZ
9067,6650-5,21320,NaN,PATRICIA HUERTAS
9068,6625-7,21694,NaN,MARISOL BAUTISTA


In [28]:
df_union_confe = df_fin.merge(
    df_produccion,
    on='Num OS',
    how='left'
)

df_union_confe = df_union_confe.rename(columns={'OP-D_x' : 'OP-D'})


In [29]:
# Filas que NO coincidieron por 'Num OS'
df_no_coincidio = df_union_confe[df_union_confe['TALLER SERVICIO'].isna()].copy()

# Filas que SÍ coincidieron (estas ya están bien)
df_coincidencias_ok = df_union_confe[df_union_confe['TALLER SERVICIO'].notna()].copy()

# Columnas que quieres reintentar traer de df_produccion
cols_a_eliminar = ['TALLER SERVICIO', 'CANTIDAD ENTREGA SATELITE'] # Ajusta según las que necesites "limpiar"

# Eliminamos las columnas vacías para que el nuevo merge las traiga frescas
df_no_coincidio = df_no_coincidio.drop(columns=cols_a_eliminar)

# Nuevo intento de unión, ahora por 'OP-D'
df_reintento = df_no_coincidio.merge(
    df_produccion,
    on='OP-D',
    how='left'
)
dfff = pd.concat([df_coincidencias_ok, df_reintento], ignore_index=True)

In [30]:
dfff.to_excel('dffinn.xlsx')